![display relevant image here](path/url/to/image)
- Banner/header image

# Title
- Relevant to Data and Business Context

## Overview
- BLUF (Bottom Line Up Front)
- One paragraph summary of findings and analysis
- Frame your 'story'

## Business Understanding
- Set the stage for analysis
- Why are these findings relevant/important?
- Introduce stakeholders
- Postulate about questions you want to ask/answer

## Data Understanding
- Present the source of data
- Describe the data available
- What is relevant to keep what is not
- Present any data cleaning that needs to happen
- Null values? Type mismatches? Duplicates?

In [1]:
# Imports
import pyspark
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType, LongType
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Work around for pyspark to sense the java dev kit
os.environ['JAVA_HOME'] = 'C:\\Program Files\\Java\\jdk-25.0.5.0.3'

In [2]:
# EDA Code Here - Create New Cells As Needed
spark = (
    SparkSession.builder
    .appName("GooglePlayStoreDataCleaning")
    .getOrCreate()
)

e:\Moringa_Data_Science\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
# confirm that the spark session is active
if (spark.sparkContext.appName == 'GooglePlayStoreDataCleaning'):
    print(f'{spark.sparkContext.appName} is running')
else:
    print(f'{spark.sparkContext.appName} is not running')

GooglePlayStoreDataCleaning is running


In [4]:
# load the dataset
play_store = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)\
    .option('quote', '"')
    .option('escape', '"')
    .option("mode", "PERMISSIVE")
    .csv('google_play_store_dataset.csv')
)

print("Original row count:", play_store.count())
print("Original column count:", len(play_store.columns))
play_store.show(5, truncate=False)
play_store.printSchema()

Original row count: 10841
Original column count: 13
+--------------------------------------------------+--------------+------+-------+----+-----------+----+-----+--------------+-------------------------+----------------+------------------+------------+
|App                                               |Category      |Rating|Reviews|Size|Installs   |Type|Price|Content Rating|Genres                   |Last Updated    |Current Ver       |Android Ver |
+--------------------------------------------------+--------------+------+-------+----+-----------+----+-----+--------------+-------------------------+----------------+------------------+------------+
|Photo Editor & Candy Camera & Grid & ScrapBook    |ART_AND_DESIGN|4.1   |159    |19M |10,000+    |Free|0    |Everyone      |Art & Design             |January 7, 2018 |1.0.0             |4.0.3 and up|
|Coloring book moana                               |ART_AND_DESIGN|3.9   |967    |14M |500,000+   |Free|0    |Everyone      |Art & Design;Preten

## Data Preparation



In [5]:
# checking for & dropping duplicates
rows_before_deduplication = play_store.count()
play_store = play_store.dropDuplicates()
rows_after_deduplication = play_store.count()

print("Exact duplicate rows removed:", rows_before_deduplication - rows_after_deduplication)
print("Rows remaining:", rows_after_deduplication)

Exact duplicate rows removed: 483
Rows remaining: 10358


In [6]:
# repairing the misaligned row
app_name = "Life Made WI-Fi Touchscreen Photo Frame"

columns_to_preserve = [
    "Category", "Rating", "Reviews", "Size", "Installs", "Type", "Price",
    "Content Rating", "Genres", "Last Updated", "Current Ver"
]

for c in columns_to_preserve:
    play_store = play_store.withColumn("_src_" + c.replace(" ", "_"), F.col(c))

bad = F.col("App") == app_name

play_store = (
    play_store
    .withColumn("Rating", F.when(bad, F.col("_src_Category")).otherwise(F.col("Rating")))
    .withColumn("Reviews", F.when(bad, F.col("_src_Rating")).otherwise(F.col("Reviews")))
    .withColumn("Size", F.when(bad, F.col("_src_Reviews")).otherwise(F.col("Size")))
    .withColumn("Installs", F.when(bad, F.col("_src_Size")).otherwise(F.col("Installs")))
    .withColumn("Type", F.when(bad, F.col("_src_Installs")).otherwise(F.col("Type")))
    .withColumn("Price", F.when(bad, F.col("_src_Type")).otherwise(F.col("Price")))
    .withColumn("Content Rating", F.when(bad, F.col("_src_Price")).otherwise(F.col("Content Rating")))
    .withColumn("Last Updated", F.when(bad, F.col("_src_Genres")).otherwise(F.col("Last Updated")))
    .withColumn("Current Ver", F.when(bad, F.col("_src_Last_Updated")).otherwise(F.col("Current Ver")))
    .withColumn("Android Ver", F.when(bad, F.col("_src_Current_Ver")).otherwise(F.col("Android Ver")))
    .withColumn("Category", F.when(bad, F.lit(None).cast("string")).otherwise(F.col("Category")))
    .withColumn("Genres", F.when(bad, F.lit(None).cast("string")).otherwise(F.col("Genres")))
)

helper_columns = [c for c in play_store.columns if c.startswith("_src_")]
play_store = play_store.drop(*helper_columns)

play_store.filter(F.col("App") == app_name).show(truncate=False)

+---------------------------------------+--------+------+-------+----+--------+----+-----+--------------+------+-----------------+-----------+-----------+
|App                                    |Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|Genres|Last Updated     |Current Ver|Android Ver|
+---------------------------------------+--------+------+-------+----+--------+----+-----+--------------+------+-----------------+-----------+-----------+
|Life Made WI-Fi Touchscreen Photo Frame|NULL    |1.9   |19     |3.0M|1,000+  |Free|0    |Everyone      |NULL  |February 11, 2018|1.0.19     |4.0 and up |
+---------------------------------------+--------+------+-------+----+--------+----+-----+--------------+------+-----------------+-----------+-----------+



In [7]:
# trimming text and standardizing empty strings
text_cols = [
    "App", "Category", "Type", "Content Rating", "Genres", "Current Ver",
    "Android Ver", "Reviews", "Size", "Installs", "Price", "Rating",
    "Last Updated"
]

for c in text_cols:
    play_store = play_store.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))
    )

play_store.select("App", "Category", "Type", "Content Rating").show(5, truncate=False)

+----------------------------------------------------------------------------------------+-------------------+----+--------------+
|App                                                                                     |Category           |Type|Content Rating|
+----------------------------------------------------------------------------------------+-------------------+----+--------------+
|Garden Coloring Book                                                                    |ART_AND_DESIGN     |Free|Everyone      |
|Used car is the first car - used car purchase, used car quotation, dealer information to|AUTO_AND_VEHICLES  |Free|Everyone      |
|Wattpad 📖 Free Books                                                                   |BOOKS_AND_REFERENCE|Free|Teen          |
|Hangouts                                                                                |COMMUNICATION      |Free|Everyone      |
|Viki: Asian TV Dramas & Movies                                                     

In [8]:
# changing the data types
play_store = (
    play_store
    .withColumn("Rating", F.col("Rating").try_cast(DoubleType()))
    .withColumn("Reviews", F.regexp_replace("Reviews", ",", "").try_cast(LongType()))
    .withColumn(
        "Installs",
        F.regexp_replace(F.regexp_replace("Installs", "[,+]", ""), ",", "").try_cast(LongType())
    )
    .withColumn(
        "Price",
        F.regexp_replace("Price", "[$,]", "").try_cast(DoubleType())
    )
)

play_store.select("Rating", "Reviews", "Installs", "Price").show(10, truncate=False)
play_store.printSchema()

+------+-------+----------+-----+
|Rating|Reviews|Installs  |Price|
+------+-------+----------+-----+
|4.4   |13791  |1000000   |0.0  |
|4.6   |5097   |1000000   |0.0  |
|4.6   |2914724|100000000 |0.0  |
|4.0   |3419513|1000000000|0.0  |
|4.3   |407719 |10000000  |0.0  |
|4.0   |196    |10000     |0.0  |
|4.7   |51517  |1000000   |0.0  |
|4.5   |9881829|100000000 |0.0  |
|4.4   |6198880|100000000 |0.0  |
|4.3   |2045554|50000000  |0.0  |
+------+-------+----------+-----+
only showing top 10 rows
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: long (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (

In [9]:
# Changing data types pt 2

# Step 1: Standardize the Size column
play_store = play_store.withColumn(
    "Size",
    F.upper(F.trim(F.col("Size").cast("string")))
)

# Step 2: Extract the numerical part
play_store = play_store.withColumn(
    "Size_Number",
    F.regexp_extract(
        F.col("Size"),
        r"([0-9]+(?:\.[0-9]+)?)",
        1
    )
)

# Step 3: Safely convert extracted values to double
play_store = play_store.withColumn(
    "Size_Number",
    F.expr("try_cast(Size_Number AS DOUBLE)")
)

# Step 4: Convert all sizes into MB
play_store = play_store.withColumn(
    "Size_MB",
    F.when(
        F.col("Size").endswith("K"),
        F.col("Size_Number") / 1024
    )
    .when(
        F.col("Size").endswith("M"),
        F.col("Size_Number")
    )
    .otherwise(None)
)

# Step 5: Remove unnecessary columns
play_store = play_store.drop("Size", "Size_Number")

# Step 6: Inspect the results
play_store.select("App", "Size_MB").show(20, truncate=False)

play_store.printSchema()

+----------------------------------------------------------------------------------------+-------+
|App                                                                                     |Size_MB|
+----------------------------------------------------------------------------------------+-------+
|Garden Coloring Book                                                                    |33.0   |
|Used car is the first car - used car purchase, used car quotation, dealer information to|19.0   |
|Wattpad 📖 Free Books                                                                   |NULL   |
|Hangouts                                                                                |NULL   |
|Viki: Asian TV Dramas & Movies                                                          |NULL   |
|50 Healthy Slow Cooker Recipes                                                          |9.0    |
|BeyondMenu Food Delivery                                                                |NULL   |
|My Talking

In [10]:
# changing data types pt 3

play_store = (
    play_store
    .withColumn("Last_Updated", F.to_date(F.col("Last Updated"), "MMMM d, yyyy"))
    .drop("Last Updated")
)

play_store.select("Last_Updated").show(10, truncate=False)

+------------+
|Last_Updated|
+------------+
|2017-09-20  |
|2018-07-23  |
|2018-08-01  |
|2018-07-21  |
|2018-07-27  |
|2017-02-17  |
|2018-08-01  |
|2018-07-03  |
|2018-07-10  |
|2018-07-30  |
+------------+
only showing top 10 rows


In [11]:
# checking the inference of the type and price column


# Step 1: Clean the Price column
play_store = play_store.withColumn(
    "Price",
    F.trim(F.col("Price").cast("string"))
)

# Step 2: Remove currency symbols and commas
play_store = play_store.withColumn(
    "Price_Clean",
    F.regexp_replace(
        F.col("Price"),
        r"[^0-9.]",
        ""
    )
)

# Step 3: Safely convert Price into a numeric value
play_store = play_store.withColumn(
    "Price",
    F.expr("try_cast(Price_Clean AS DOUBLE)")
)

# Step 4: Remove temporary column
play_store = play_store.drop("Price_Clean")

# Step 5: Handle missing Type values
play_store = play_store.withColumn(
    "Type",
    F.when(
        F.col("Type").isNull() & (F.col("Price") == 0),
        "Free"
    )
    .when(
        F.col("Type").isNull() & (F.col("Price") > 0),
        "Paid"
    )
    .otherwise(F.col("Type"))
)

# Step 6: Inspect results
play_store.select("Price", "Type").show(20, truncate=False)

play_store.printSchema()

+-----+----+
|Price|Type|
+-----+----+
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|1.5  |Paid|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
|0.0  |Free|
+-----+----+
only showing top 20 rows
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: long (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)
 |-- Size_MB: double (nullable = true)
 |-- Last_Updated: date (nullable = true)



In [12]:
# validating the rating column
play_store = play_store.withColumn(
    "Rating",
    F.when((F.col("Rating") < 0) | (F.col("Rating") > 5), None)
     .otherwise(F.col("Rating"))
)


In [13]:
# dealing with nulls
play_store = (
    play_store
    .withColumn("Current Ver", F.coalesce(F.col("Current Ver"), F.lit("Unknown")))
    .withColumn("Android Ver", F.coalesce(F.col("Android Ver"), F.lit("Unknown")))
)

In [14]:
# final quality check
final_row_count = play_store.count()
final_column_count = len(play_store.columns)


print("Final rows:", final_row_count)
print("Final columns:", final_column_count)



play_store.printSchema()
play_store.show(5, truncate=False)

Final rows: 10358
Final columns: 13
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: long (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Current Ver: string (nullable = false)
 |-- Android Ver: string (nullable = false)
 |-- Size_MB: double (nullable = true)
 |-- Last_Updated: date (nullable = true)

+----------------------------------------------------------------------------------------+-------------------+------+-------+----------+----+-----+--------------+-----------------+------------------+------------------+-------+------------+
|App                                                                                     |Category           |Rating|Reviews|Installs  |Type|Price|Content Rating|Genres           |Current Ver       |Andro

In [15]:
# export the cleaned data

output_path = "google_play_store_cleaned_spark"

play_store.toPandas().to_csv('play_store_cleaned.csv', index=False, encoding='utf-8')

e:\Moringa_Data_Science\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Data Analysis

In [16]:
# Analysis Code Here - if needed
category_summary = (
    play_store.filter(F.col('Category').isNotNull())
    .groupBy("Category")
    .agg(
        F.count("App").alias("Total_Apps"),
        F.sum("Installs").alias("Total_Installs"),
        F.round(F.avg("Rating"), 2).alias("Avg_Rating"),
        F.sum("Reviews").alias("Total_Reviews"),
        F.round(F.avg("Size_MB"), 1).alias("Avg_Size_MB")
    )
    .orderBy(F.desc("Total_Installs"))
)

category_summary.show(truncate=False)

category_summary.toPandas().to_csv('Category Summary.csv', index=False, encoding='utf-8')

+-------------------+----------+--------------+----------+-------------+-----------+
|Category           |Total_Apps|Total_Installs|Avg_Rating|Total_Reviews|Avg_Size_MB|
+-------------------+----------+--------------+----------+-------------+-----------+
|GAME               |1121      |31544024415   |4.28      |1415536650   |44.1       |
|COMMUNICATION      |366       |24152276251   |4.15      |601273552    |11.7       |
|SOCIAL             |280       |12513867902   |4.25      |533576829    |16.9       |
|PRODUCTIVITY       |407       |12463091369   |4.2       |102554498    |12.9       |
|TOOLS              |843       |11452771915   |4.05      |273185044    |8.8        |
|FAMILY             |1943      |10041692505   |4.19      |396771969    |27.9       |
|PHOTOGRAPHY        |322       |9721247655    |4.18      |204297410    |16.8       |
|TRAVEL_AND_LOCAL   |237       |6361887146    |4.09      |55565160     |24.5       |
|VIDEO_PLAYERS      |175       |6222002720    |4.06      |1103801

e:\Moringa_Data_Science\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [17]:
spark.stop()

### Link to Published Dashboard

The link to the tableau Public is <br> <https://public.tableau.com/views/Book1_17947552839390/Relationshipbetweensizeandrating?:language=en-GB&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link>

## Conclusion



### Challenges
1. Cleaning of the google play dataset was difficult due to some apps having commas and the CSV uses commas as delimiters. The solution was to use the following code:<br>```spark.read.option("header", True).option("inferSchema", False).option('quote', '"').option('escape', '"').option("mode", "PERMISSIVE").csv('google_play_store_dataset.csv')```. The option function helps in making it easier for cleaning.
2. Loading of Pyspark In windows is difficult as the java System Variable makes is not automatic. My work around is to import the os module and set the path of Java by use of the code `os.environ['JAVA_HOME'] = 'C:\\Program Files\\Java\\jdk-25.0.5.0.3'` which makes it easier for pyspark to read the path.
3. Saving of the pyspark file came with 2 problems: one is the lack of hadoop for saving which i had to make the hadoop variable, the second is that the file that is saved via hadoop was difficult to handle and my solution was to turn it to pandas then save it to csv. 

### Findings
1. The **Family** categories of apps has the highest number of distinct apps *(1908)* but has fewer number of installs *(around 10 billion)* compared to the **Game** categories of apps which has the second highest number of distinct apps *(960)* with the highest number of installs *(around 32 billion)*
2. **Events** is the category with the highest rating but the fewest number of installs. This leads me to concude that most Events apps are not made all year round.
3. *Game* apps have the largest number of installs but are pretty popular. This is due to many of the games have a freemium model of generating money.
4. Size of the apps and the avarge rating are not affected by the type of app payment but the time that it takes for updates, especially for games are important.
5. There are upto around *7.83%* apps that are paid for upfront.
6. Most apps have a rating of *4.0*

### Advice for the shareholders
- I would advice on them investing into games with less than 50.0MB of space and to be either priced on a one time pay or go through the micro-transaction route through battle passes and in-currency bundles.